# BestShot — Provision Training Environment on Chameleon

This notebook sets up a GPU server on Chameleon for BestShot model training.
It is adapted from the `llm-chi` lab (`2_create_server.ipynb`).

**Before running this notebook:**
- Make sure you have an active lease on CHI@TACC with a gpu_mi100 node reservation
- Make sure your keypair is registered on Chameleon
- Run this notebook from the Chameleon JupyterHub environment

**What this notebook does:**
1. Connects to your existing lease
2. Brings up the GPU server from a boot volume
3. Sets up security groups and floating IP
4. Installs Docker + ROCm container toolkit
5. Clones the BestShot repo
6. Downloads KonIQ-10k dataset
7. Starts MLflow (via Docker Compose)
8. Builds the training Docker container
9. Runs a training job

## Step 1 — Connect to Chameleon project and site

In [ ]:
from chi import server, context, lease, network
import chi, os, time

context.version = "1.0"
context.choose_project()
context.choose_site(default="CHI@TACC")

## Step 2 — Get your existing lease

Replace `bestshot-train-proj19` with the exact name of the lease you created in the Chameleon GUI.
The status should show as **ACTIVE**.

In [ ]:
# Replace with your actual lease name (must end in proj19)
LEASE_NAME = "train_test_proj19"

l = lease.get_lease(LEASE_NAME)
l.show()

## Step 3 — Bring up the server

For bare metal nodes at CHI@TACC, we use a node reservation directly (not a flavor).
The image is `CC-Ubuntu22.04` — use Ubuntu 22.04 for best ROCm compatibility.

In [ ]:
username = os.getenv('USER')
server_name = f"bestshot-gpu-proj19"

try:
    s = server.get_server(server_name)
    print(f"Server {server_name} already exists. Skipping create.")
except Exception:
    # Get the node reservation ID from the lease
    node_reservation_id = lease.get_node_reservation(l["id"])

    s = server.Server(
        server_name,
        reservation_id=node_reservation_id,
        image_name="CC-Ubuntu22.04",
    )
    s.submit(idempotent=True)
    print(f"Server {server_name} is ready.")

## Step 4 — Security groups

Open SSH (22) and MLflow UI (8000) ports.

In [ ]:
security_groups = [
    {'name': 'allow-ssh',  'port': 22,   'description': 'Enable SSH on port 22'},
    {'name': 'allow-8000', 'port': 8000, 'description': 'Enable MLflow UI on port 8000'},
    {'name': 'allow-9000', 'port': 9000, 'description': 'Enable MinIO on port 9000'},
]

for sg in security_groups:
    secgroup = network.SecurityGroup({
        'name': sg['name'],
        'description': sg['description'],
    })
    secgroup.add_rule(direction='ingress', protocol='tcp', port=sg['port'])
    secgroup.submit(idempotent=True)
    s.add_security_group(sg['name'])

print(f"Security groups applied: {[sg['name'] for sg in security_groups]}")

## Step 5 — Associate floating IP and verify connectivity

In [ ]:
s.associate_floating_ip()

In [ ]:
s.refresh()
s.check_connectivity()

In [ ]:
# Note the floating IP — you'll use it to access MLflow in your browser
s.refresh()
s.show(type="widget")

## Step 6 — Install Docker + ROCm container toolkit

For AMD gpu_mi100 nodes we install ROCm instead of NVIDIA container toolkit.

In [ ]:
s.execute("curl -sSL https://get.docker.com | sudo sh")

In [ ]:
s.execute("sudo groupadd -f docker && sudo usermod -aG docker $USER")

In [ ]:
# Install ROCm container runtime for AMD GPU access
s.execute("""
sudo apt-get update && \
sudo apt-get install -y wget gnupg && \
wget -q -O - https://repo.radeon.com/rocm/rocm.gpg.key | sudo apt-key add - && \
echo 'deb [arch=amd64] https://repo.radeon.com/amdgpu/latest/ubuntu focal main' | sudo tee /etc/apt/sources.list.d/amdgpu.list && \
echo 'deb [arch=amd64] https://repo.radeon.com/rocm/apt/6.2 focal main' | sudo tee /etc/apt/sources.list.d/rocm.list && \
sudo apt-get update && \
sudo apt-get install -y rocm-hip-runtime
""")

In [ ]:
# Verify AMD GPU is visible — should show MI100 info
s.execute("docker run --rm --device=/dev/kfd --device=/dev/dri --group-add video rocm/rocm-terminal rocm-smi")

## Step 7 — Clone the BestShot repo

In [ ]:
# Replace with your team's actual GitHub repo URL
REPO_URL = "https://github.com/anokhimehta/bestshot"

s.execute(f"git clone {REPO_URL} ~/bestshot")

## Step 8 — Download KonIQ-10k dataset

Downloading directly to the Chameleon instance.
Using 512x384 images (~767 MB) — sufficient for EfficientNet-B3 training.

Source: University of Konstanz VQA group (https://database.mmsp-kn.de/koniq-10k-database.html)

In [ ]:
s.execute("mkdir -p ~/data/koniq10k")

In [ ]:
# Download images (512x384) — ~767 MB, takes a few minutes
s.execute("""
wget -q --show-progress \
    -P ~/data/koniq10k \
    http://datasets.vqa.mmsp-kn.de/archives/koniq10k_512x384.zip
""")

In [ ]:
# Download scores CSV — ~304 KB
s.execute("""
wget -q --show-progress \
    -P ~/data/koniq10k \
    http://datasets.vqa.mmsp-kn.de/archives/koniq10k_scores_and_distributions.zip
""")

In [ ]:
# Unzip both
s.execute("""
cd ~/data/koniq10k && \
unzip -q koniq10k_512x384.zip && \
unzip -q koniq10k_scores_and_distributions.zip
""")

In [ ]:
# Verify — should show 10,073 images and the scores CSV
s.execute("ls ~/data/koniq10k/ && echo '---' && ls ~/data/koniq10k/512x384/ | wc -l")

## Step 9 — Set MLflow tracking URI

MLflow runs on a separate VM provisioned by `mlflow_setup.ipynb`.
Paste the floating IP of that server here before running training.

In [ ]:
# Paste the floating IP from mlflow_setup.ipynb here
MLFLOW_IP = ""  # e.g. "129.114.26.99"

MLFLOW_TRACKING_URI = f"http://{MLFLOW_IP}:8000"
print(f"MLflow tracking URI: {MLFLOW_TRACKING_URI}")

## Step 10 — Build the training container

Builds the Docker image from `training/Dockerfile` and `training/requirements.txt` in the repo.
This takes a few minutes the first time — subsequent builds are faster due to layer caching.

In [ ]:
# Build the training container image
# The build context is training/ so COPY requirements.txt works correctly
s.execute("docker build -t bestshot-train:latest ~/bestshot/training/")

In [ ]:
# Verify the image was built successfully
s.execute("docker images bestshot-train")

## Step 11 — Run a training job

Runs `train.py` inside the container with:
- `--gpus all` so the container can see the GPU
- KonIQ-10k data mounted from the host into `/data/koniq10k` inside the container
- `MLFLOW_TRACKING_URI` pointing at the separate MLflow server from Step 9
- Repo mounted so we can edit `train.py` without rebuilding the image each time

Replace `--config config/baseline.yaml` with whichever config you want to run.

In [ ]:
s.execute(f"""
docker run --rm \\
    --device=/dev/kfd \\
    --device=/dev/dri \\
    --group-add video \\
    -v ~/data/koniq10k:/data/koniq10k \\
    -v ~/bestshot:/workspace \\
    -w /workspace \\
    -e MLFLOW_TRACKING_URI={MLFLOW_TRACKING_URI} \\
    bestshot-train:latest \\
    python training/train.py --config training/config/baseline.yaml
""")

## Done!
Environment is fully set up. Once training finishes:

1. Open the MLflow UI at the URL printed in Step 9 to see run
2. Check metrics (MSE, MAE, PLCC) and training time were logged
3. Run additional candidates by changing the config file passed to `--config`